In [1]:
import pathlib
from argparse import ArgumentParser
import yaml
import torch
import torchmetrics

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

/home/jcruse/.conda/envs/jcruse_env/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation

In [3]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [4]:
config['data']['corpus']

{'root': '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'}

In [5]:
model = AttentionalTrackingModule(config)
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=model.audio_transforms)

In [6]:
parent_path = '/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/'
ckpt_path = parent_path + 'attn_cue_models/attn_cue_jsin_pilot_no_pretrain_bs_64_lr_1e-4/checkpoints/epoch=1-step=120790.ckpt'

ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])

<All keys matched successfully>

In [7]:
fg_acc = torchmetrics.Accuracy()
bg_acc = torchmetrics.Accuracy()

In [8]:
model = model.eval().cuda()
fg_acc.cuda()
bg_acc.cuda()

Accuracy()

In [9]:
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

/home/jcruse/.conda/envs/jcruse_env/lib/python3.9/site-packages/torch/utils/data/dataloader.py:487: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [10]:
from tqdm import tqdm

In [11]:
for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


  0%|          | 0/1050 [02:15<?, ?it/s]


RuntimeError: DataLoader worker (pid 11231) is killed by signal: Killed. 

In [ ]:
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

In [ ]:
f_word

In [ ]:
fg_out.log_softmax(-1).argmax(-1)

In [ ]:
b_word

In [ ]:
bg_out.log_softmax(-1).argmax(-1)

## Eval when talkers at same SNR

In [ ]:
import src.audio_transforms as at


audio_config = config['data']['audio']
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

In [ ]:
no_sep_dataet = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)

In [ ]:
no_sep_val_loader = torch.utils.data.DataLoader(no_sep_dataet, batch_size=4, num_workers=10)

In [ ]:
fg_acc = torchmetrics.Accuracy()
bg_acc = torchmetrics.Accuracy()

In [ ]:
model = model.eval().cuda()
fg_acc.cuda()
bg_acc.cuda()

In [ ]:
for ix, batch in tqdm(enumerate(no_sep_val_loader), total=len(no_sep_val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    f_acc = fg_acc(fg_out, f_word)
    b_acc = bg_acc(bg_out, b_word)
    
    if ix % 100 == 0:
        print(f"Foreground accuracy: {f_acc}")
        print(f"Background accuracy: {b_acc}")
        
    if ix == len(no_sep_val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


In [ ]:
# Final performance
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
fig, 

plt.imshow(mix.cpu().squeeze()[0], aspect='auto')

In [ ]:
plt.imshow(f_cue.cpu().squeeze()[0], aspect='auto')

In [ ]:
plt.imshow(b_cue.cpu().squeeze()[0], aspect='auto')